# MRI Data Preparation Pipeline
### ds003047 · sub-02–sub-11 · T1w + DWI
**Steps:** Resampling → Pad/Crop → Brain Masking (SynthStrip) → Patch2Self Denoising → Patch Extraction → Train/Val/Test Split

## 0. Imports & Configuration

In [ ]:
import os, sys, json, shutil, warnings, urllib.request
from pathlib import Path
import subprocess

# install missing packages silently
for _p in ['nibabel','dipy','pandas','tqdm','matplotlib','scipy']:
    try: __import__(_p)
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install',_p,'-q'])

import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import zoom as sp_zoom
from tqdm import tqdm
from dipy.io.image import load_nifti, save_nifti
from dipy.io.gradients import read_bvals_bvecs
from dipy.denoise.patch2self import patch2self

# ── USER CONFIG ───────────────────────────────────────────────────────────────
BIDS_ROOT   = Path('/Users/test/ds003047')
OUTPUT_ROOT = Path('/Users/test/ds003047/prepared')
SUBJECTS    = [f'sub-{i:02d}' for i in range(2, 12)]
SESSIONS    = ['ses-1', 'ses-2']

TARGET_VOX   = (2.0, 2.0, 2.0)    # isotropic target resolution (mm)
TARGET_SHAPE = (96, 128, 128)      # final shape after pad/crop (divisible by 8)
PATCH_SIZE   = (12, 12, 12)
PATCH_STRIDE = (6, 6, 6)
MAX_PATCHES  = 2000
RANDOM_SEED  = 42

for d in ['resampled','masks','augmented','qc']:
    (OUTPUT_ROOT/d).mkdir(parents=True, exist_ok=True)

print('✓ Ready')
print(f'  BIDS:   {BIDS_ROOT}')
print(f'  Output: {OUTPUT_ROOT}')

## 1. Dataset Discovery

In [ ]:
records = []
for sub in SUBJECTS:
    for ses in SESSIONS:
        base = BIDS_ROOT/sub/ses
        if not base.exists(): continue
        for t1 in (base/'anat').glob('*_T1w.nii.gz'):
            records.append(dict(subject=sub, session=ses, modality='T1w',
                                path=str(t1), original_path=str(t1),
                                bval=None, original_bval=None))
        for dwi in (base/'dwi').glob('*_dwi.nii.gz'):
            stem = dwi.name.replace('.nii.gz','')
            bval = dwi.parent/f'{stem}.bval'
            records.append(dict(subject=sub, session=ses, modality='DWI',
                                path=str(dwi), original_path=str(dwi),
                                bval=str(bval) if bval.exists() else None,
                                original_bval=str(bval) if bval.exists() else None))

df = pd.DataFrame(records)
print(f'Found {len(df)} files  |  {dict(df.modality.value_counts())}')
df.head(8)

## 2. Resampling + Pad/Crop

In [ ]:
def resample_nib(img, target_vox):
    data = img.get_fdata(dtype=np.float32)
    cur  = np.array(img.header.get_zooms()[:3], dtype=float)
    zf   = cur / np.array(target_vox)
    if data.ndim == 4:
        vols = []
        for v in range(data.shape[-1]):
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                vols.append(sp_zoom(data[...,v], zf, order=3, prefilter=True))
        out = np.stack(vols, -1)
    else:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            out = sp_zoom(data, zf, order=3, prefilter=True)
    aff = img.affine.copy()
    aff[:3,:3] = img.affine[:3,:3] / zf
    new = nib.Nifti1Image(out, aff, img.header)
    new.header.set_zooms(tuple(target_vox) + img.header.get_zooms()[3:])
    return new

def pad_crop_nib(img, tgt):
    data = img.get_fdata(dtype=np.float32)
    is4d = data.ndim == 4
    pads, crops, shift = [], [], np.zeros(3)
    for d,(cur,t) in enumerate(zip(data.shape[:3], tgt)):
        if cur < t:
            p = t-cur; pads.append((p//2, p-p//2)); crops.append(slice(None))
        elif cur > t:
            s = (cur-t)//2; pads.append((0,0)); crops.append(slice(s,s+t)); shift[d]=s
        else:
            pads.append((0,0)); crops.append(slice(None))
    data = data[crops[0],crops[1],crops[2],...] if is4d else data[crops[0],crops[1],crops[2]]
    data = np.pad(data, pads+[(0,0)] if is4d else pads)
    aff = img.affine.copy()
    aff[:3,3] += img.affine[:3,:3] @ shift
    return nib.Nifti1Image(data, aff, img.header)

FORCE_RERUN = False
resample_log = []

for _,row in tqdm(df.iterrows(), total=len(df), desc='Resampling'):
    sub,ses,mod = row.subject, row.session, row.modality
    out_dir = OUTPUT_ROOT/'resampled'/sub/ses/mod.lower()
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir/Path(row.original_path).name
    entry = dict(subject=sub, session=ses, modality=mod, output_path=str(out))
    if out.exists() and not FORCE_RERUN:
        img = nib.load(str(out)); orig = nib.load(row.original_path)
        entry.update(status='success',
                     original_shape=orig.shape[:3],
                     original_zooms_mm=tuple(np.round(orig.header.get_zooms()[:3],3)),
                     final_shape=img.shape[:3])
        print(f'  [skip] {sub}/{ses} {mod}')
    else:
        try:
            img = nib.load(row.original_path)
            os = img.shape[:3]; oz = tuple(np.round(img.header.get_zooms()[:3],3))
            img = resample_nib(img, TARGET_VOX)
            img = pad_crop_nib(img, TARGET_SHAPE)
            nib.save(img, str(out))
            if mod=='DWI' and row.bval:
                for ext in ['.bval','.bvec']:
                    src = Path(row.original_path.replace('.nii.gz','') + ext)
                    if src.exists(): shutil.copy2(src, out_dir/src.name)
            entry.update(status='success', original_shape=os,
                         original_zooms_mm=oz, final_shape=img.shape[:3])
            print(f'  ✓ {sub}/{ses} {mod}  {os}@{oz}mm → {img.shape[:3]}')
        except Exception as e:
            entry['status'] = f'error: {e}'; print(f'  ✗ {sub}/{ses} {mod}: {e}')
    resample_log.append(entry)

df_rs = pd.DataFrame(resample_log)
# redirect df paths to resampled files
df['path'] = df.apply(lambda r: str(
    OUTPUT_ROOT/'resampled'/r.subject/r.session/r.modality.lower()/Path(r.original_path).name), axis=1)
print('\nResampling:'); print(df_rs.status.value_counts())

In [ ]:
# ── QC: Resampling — 3 Subjects, Original vs Resampled (3 Ebenen) ────────────
ok_rs = df_rs[df_rs.status == 'success']
t1_ok = ok_rs[ok_rs.modality == 'T1w'].head(3)

if t1_ok.empty:
    print('⚠ Keine resampled T1w Dateien')
else:
    n = len(t1_ok)
    fig, axes = plt.subplots(n * 2, 3, figsize=(13, n * 4.5))
    fig.suptitle('Resampling QC — Original (oben) vs Resampled (unten)',
                 fontweight='bold', fontsize=12)

    for si, (_, row) in enumerate(t1_ok.iterrows()):
        orig = nib.load(row.get('original_path', row.output_path)
                        if hasattr(row, 'get') else row['output_path']).get_fdata(dtype=np.float32)
        rsmp = nib.load(row['output_path']).get_fdata(dtype=np.float32)
        if orig.ndim == 4: orig = orig[..., 0]
        if rsmp.ndim == 4: rsmp = rsmp[..., 0]

        planes_o = [orig[orig.shape[0]//2, :, :],
                    orig[:, orig.shape[1]//2, :],
                    orig[:, :, orig.shape[2]//2]]
        planes_r = [rsmp[rsmp.shape[0]//2, :, :],
                    rsmp[:, rsmp.shape[1]//2, :],
                    rsmp[:, :, rsmp.shape[2]//2]]
        pnames = ['Sagittal', 'Coronal', 'Axial']
        vmin = 0; vmax = np.percentile(orig[orig > 0], 99) if orig.any() else 1

        for col, (po, pr, pn) in enumerate(zip(planes_o, planes_r, pnames)):
            ro, rr = si * 2, si * 2 + 1
            axes[ro, col].imshow(np.rot90(po), cmap='gray', vmin=vmin, vmax=vmax)
            axes[rr, col].imshow(np.rot90(pr), cmap='gray', vmin=vmin, vmax=vmax)
            if col == 0:
                axes[ro, 0].set_ylabel(
                    f"{row['subject']} {row['session']}\nOrig {row.get('original_shape','')}\n"
                    f"@ {row.get('original_zooms_mm','')}mm",
                    fontsize=7, rotation=0, labelpad=80, va='center')
                axes[rr, 0].set_ylabel(
                    f"Resampled {row.get('final_shape','')}\n@ {TARGET_VOX}mm",
                    fontsize=7, rotation=0, labelpad=80, va='center')
            if si == 0:
                axes[ro, col].set_title(pn, fontsize=9, fontweight='bold')
            for ax in [axes[ro, col], axes[rr, col]]:
                ax.axis('off')

    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT/'qc'/'resample_qc.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ Resampling QC gespeichert')


## 3. Brain Masking with SynthStrip

Runs **entirely in Python using nibabel + torch** — no surfa, no subprocess, no shell binary.

In [ ]:
# ── 1. torch ──────────────────────────────────────────────────────────────────
try:
    import torch
    print(f'✓ torch {torch.__version__}')
except ImportError:
    subprocess.check_call([sys.executable,'-m','pip','install','torch','-q'])
    import torch; print('✓ torch installed')

import torch.nn as nn

# ── 2. Define StripModel and ConvBlock directly ───────────────────────────────
# Copied verbatim from the mri_synthstrip script (MIT licence).
# No surfa, no argparse, no CLI — pure torch classes only.

class ConvBlock(nn.Module):
    def __init__(self, ndims, in_channels, out_channels, stride=1, activation='leaky'):
        super().__init__()
        Conv = getattr(nn, f'Conv{ndims}d')
        self.conv = Conv(in_channels, out_channels, 3, stride, 1)
        self.activation = nn.LeakyReLU(0.2) if activation == 'leaky' else None
    def forward(self, x):
        out = self.conv(x)
        return self.activation(out) if self.activation else out

class StripModel(nn.Module):
    def __init__(self, nb_features=16, nb_levels=7, feat_mult=2, max_features=64,
                 nb_conv_per_level=2, max_pool=2, return_mask=False):
        super().__init__()
        ndims = 3
        feats = np.round(nb_features * feat_mult ** np.arange(nb_levels)).astype(int)
        feats = np.clip(feats, 1, max_features)
        nb_features = [np.repeat(feats[:-1], nb_conv_per_level),
                       np.repeat(np.flip(feats), nb_conv_per_level)]
        enc_nf, dec_nf = nb_features
        nb_dec_convs   = len(enc_nf)
        final_convs    = dec_nf[nb_dec_convs:]
        dec_nf         = dec_nf[:nb_dec_convs]
        self.nb_levels = int(nb_dec_convs / nb_conv_per_level) + 1
        max_pool       = [max_pool] * self.nb_levels
        MaxPooling     = getattr(nn, f'MaxPool{ndims}d')
        self.pooling    = [MaxPooling(s) for s in max_pool]
        self.upsampling = [nn.Upsample(scale_factor=s, mode='nearest') for s in max_pool]
        prev_nf = 1; encoder_nfs = [prev_nf]
        self.encoder = nn.ModuleList()
        for level in range(self.nb_levels - 1):
            convs = nn.ModuleList()
            for conv in range(nb_conv_per_level):
                nf = enc_nf[level * nb_conv_per_level + conv]
                convs.append(ConvBlock(ndims, prev_nf, nf)); prev_nf = nf
            self.encoder.append(convs); encoder_nfs.append(prev_nf)
        encoder_nfs = np.flip(encoder_nfs)
        self.decoder = nn.ModuleList()
        for level in range(self.nb_levels - 1):
            convs = nn.ModuleList()
            for conv in range(nb_conv_per_level):
                nf = dec_nf[level * nb_conv_per_level + conv]
                convs.append(ConvBlock(ndims, prev_nf, nf)); prev_nf = nf
            self.decoder.append(convs)
            if level < (self.nb_levels - 1): prev_nf += encoder_nfs[level]
        self.remaining = nn.ModuleList()
        for nf in final_convs:
            self.remaining.append(ConvBlock(ndims, prev_nf, nf)); prev_nf = nf
        self.remaining.append(ConvBlock(ndims, prev_nf, 2 if return_mask else 1, activation=None))
        if return_mask: self.remaining.append(nn.Softmax(dim=1))
    def forward(self, x):
        x_history = [x]
        for level, convs in enumerate(self.encoder):
            for conv in convs: x = conv(x)
            x_history.append(x); x = self.pooling[level](x)
        for level, convs in enumerate(self.decoder):
            for conv in convs: x = conv(x)
            if level < (self.nb_levels - 1):
                x = self.upsampling[level](x)
                x = torch.cat([x, x_history.pop()], dim=1)
        for conv in self.remaining: x = conv(x)
        return x

print('✓ StripModel defined')

# ── 3. Download weights ───────────────────────────────────────────────────────
MODEL_DIR  = Path.home()/'.cache'/'freesurfer'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR/'synthstrip.1.pt'
if not MODEL_PATH.exists():
    print('Downloading SynthStrip weights (29 MB)...')
    urllib.request.urlretrieve(
        'https://surfer.nmr.mgh.harvard.edu/docs/synthstrip/requirements/synthstrip.1.pt',
        MODEL_PATH)
print(f'✓ Weights: {MODEL_PATH}')

# ── 4. Load model ─────────────────────────────────────────────────────────────
_SS_MODEL = StripModel()
ckpt = torch.load(str(MODEL_PATH), map_location='cpu')
# weights are nested under model_state_dict in the .pt file
weights = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
_SS_MODEL.load_state_dict(weights)
_SS_MODEL.eval()
print('✓ SynthStrip model loaded and ready')

# ── 5. Pure-nibabel masking function ─────────────────────────────────────────
def _conform(img, voxsize=1.0):
    """Resample to isotropic voxsize using scipy (no surfa)."""
    data = img.get_fdata(dtype=np.float32)
    if data.ndim == 4: data = data[..., 0]
    cur = np.array(img.header.get_zooms()[:3], dtype=float)
    zf  = cur / voxsize
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        out = sp_zoom(data, zf, order=1)
    aff = img.affine.copy(); aff[:3,:3] = img.affine[:3,:3] / zf
    return nib.Nifti1Image(out.astype(np.float32), aff)

def _resample_mask(mask_img, ref_img):
    """Nearest-neighbour resample mask back to ref_img space."""
    tgt = ref_img.shape[:3]; src = mask_img.shape[:3]
    zf  = np.array(tgt) / np.array(src)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        out = sp_zoom(mask_img.get_fdata(dtype=np.float32), zf, order=0)
    result = np.zeros(tgt, np.float32)
    s = tuple(slice(0, min(a,b)) for a,b in zip(tgt, out.shape))
    result[s] = out[s]
    return nib.Nifti1Image(result, ref_img.affine)

from scipy.ndimage import binary_dilation, binary_fill_holes, generate_binary_structure
import nibabel.orientations as nio

def reorient_to_ras(img):
    orig_ornt = nio.io_orientation(img.affine)
    ras_ornt  = nio.axcodes2ornt(('R', 'A', 'S'))
    return img.as_reoriented(nio.ornt_transform(orig_ornt, ras_ornt))

def _refine_mask(mask_arr, dilation_mm=4, voxsize=1.0):
    """Fill holes then dilate to recover cortex near skull boundary."""
    filled  = binary_fill_holes(mask_arr > 0.5)
    radius  = max(1, int(round(dilation_mm / voxsize)))
    struct  = generate_binary_structure(3, 1)
    dilated = filled.copy()
    for _ in range(radius):
        dilated = binary_dilation(dilated, structure=struct)
    return dilated.astype(np.float32)

def _pad64(arr):
    pads = []
    for s in arr.shape:
        rem = s % 64
        pads.append((0, 0 if rem == 0 else 64 - rem))
    return np.pad(arr, pads), [p for _, p in pads]

def _unpad64(arr, pad_after):
    slices = tuple(slice(None,-p) if p>0 else slice(None) for p in pad_after)
    return arr[slices]

import nibabel.orientations as nio
from scipy.ndimage import (binary_dilation, binary_fill_holes,
                            binary_erosion, generate_binary_structure,
                            label as nd_label)

def reorient_to_ras(img):
    orig_ornt = nio.io_orientation(img.affine)
    ras_ornt  = nio.axcodes2ornt(('R','A','S'))
    return img.as_reoriented(nio.ornt_transform(orig_ornt, ras_ornt))

def _keep_largest(mask):
    labeled, n = nd_label(mask)
    if n == 0: return mask
    sizes   = [(labeled==i).sum() for i in range(1,n+1)]
    largest = np.argmax(sizes) + 1
    return (labeled==largest).astype(np.float32)

def _refine_mask(mask_arr, dilation_mm=5, voxsize=1.0):
    clean   = _keep_largest(mask_arr > 0.5)
    filled  = binary_fill_holes(clean)
    radius  = max(1, int(round(dilation_mm/voxsize)))
    struct  = generate_binary_structure(3,1)
    dilated = filled.copy()
    for _ in range(radius):
        dilated = binary_dilation(dilated, structure=struct)
    smoothed = binary_erosion(dilated, structure=struct)
    return smoothed.astype(np.float32)

def synthstrip_mask(input_path, output_brain, output_mask, border_mm=3):
    orig     = nib.load(str(input_path))
    orig_ras = reorient_to_ras(orig)
    conf     = _conform(orig_ras, voxsize=1.0)
    arr      = conf.get_fdata(dtype=np.float32)
    arr      = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
    arr_pad, pad_after = _pad64(arr)
    with torch.no_grad():
        sdt_pad = _SS_MODEL(torch.from_numpy(arr_pad[None,None]).float()).squeeze().numpy()
    sdt      = _unpad64(sdt_pad, pad_after)
    raw_mask = (sdt < border_mm).astype(np.float32)
    refined  = _refine_mask(raw_mask, dilation_mm=5, voxsize=1.0)
    mask_conf  = nib.Nifti1Image(refined, conf.affine)
    mask_ras   = _resample_mask(mask_conf, orig_ras)
    orig_ornt  = nio.io_orientation(orig.affine)
    ras_ornt   = nio.axcodes2ornt(('R','A','S'))
    back_trans = nio.ornt_transform(ras_ornt, orig_ornt)
    mask_orig  = mask_ras.as_reoriented(back_trans)
    d = orig.get_fdata(dtype=np.float32)
    m = mask_orig.get_fdata(dtype=np.float32)
    stripped = d * m[...,np.newaxis] if d.ndim==4 else d*m
    nib.save(mask_orig, str(output_mask))
    nib.save(nib.Nifti1Image(stripped, orig.affine, orig.header), str(output_brain))

)

In [ ]:
# ── b0 extraction + masking loop ─────────────────────────────────────────────
def extract_b0(dwi_path, bval_path, out_path, b0_thr=50):
    data, aff = load_nifti(str(dwi_path))
    bvals, _  = read_bvals_bvecs(str(bval_path), None)
    idx = np.where(bvals <= b0_thr)[0]
    if len(idx) == 0: raise ValueError('No b0 found')
    save_nifti(str(out_path), data[...,idx].mean(-1), aff)
    return idx

masking_log = []

for sub in tqdm(SUBJECTS, desc='Brain masking'):
    for ses in SESSIONS:
        rs = OUTPUT_ROOT/'resampled'/sub/ses
        if not rs.exists(): print(f'  ⚠ {sub}/{ses} not resampled yet'); continue
        od = OUTPUT_ROOT/'masks'/sub/ses; od.mkdir(parents=True, exist_ok=True)

        for t1 in (rs/'t1w').glob('*_T1w.nii.gz'):
            stem  = t1.name.replace('.nii.gz','')
            mask  = od/f'{stem}_mask.nii.gz'
            brain = od/f'{stem}_brain.nii.gz'
            if mask.exists():
                print(f'  [skip] {sub}/{ses} T1w'); status='success'
            else:
                try:
                    synthstrip_mask(t1, brain, mask)
                    status='success'; print(f'  ✓ {sub}/{ses} T1w')
                except Exception as e:
                    status=f'error: {e}'; print(f'  ✗ {sub}/{ses} T1w: {e}')
            masking_log.append(dict(subject=sub,session=ses,modality='T1w',
                status=status,mask_path=str(mask),img_path=str(t1)))

        for dwi in (rs/'dwi').glob('*_dwi.nii.gz'):
            stem = dwi.name.replace('.nii.gz','')
            mask = od/f'{stem}_mask.nii.gz'; b0 = od/f'{stem}_b0mean.nii.gz'
            brain= od/f'{stem}_b0mean_brain.nii.gz'
            orig_row = df[(df.subject==sub)&(df.session==ses)&(df.modality=='DWI')]
            if orig_row.empty or not orig_row.iloc[0].original_bval:
                print(f'  ✗ {sub}/{ses} DWI: bval missing'); continue
            bval_path = Path(orig_row.iloc[0].original_bval)
            if mask.exists():
                print(f'  [skip] {sub}/{ses} DWI'); status='success'
            else:
                try:
                    extract_b0(dwi, bval_path, b0)
                    synthstrip_mask(b0, brain, mask, border_mm=2)
                    status='success'; print(f'  ✓ {sub}/{ses} DWI')
                except Exception as e:
                    status=f'error: {e}'; print(f'  ✗ {sub}/{ses} DWI: {e}')
            masking_log.append(dict(subject=sub,session=ses,modality='DWI',
                status=status,mask_path=str(mask),img_path=str(dwi)))

df_masking = pd.DataFrame(masking_log)
print('\nMasking summary:'); print(df_masking.status.value_counts())

In [ ]:
# ── QC: Brain Masking — alle Subjects, T1w + DWI, 5 axiale Slices ────────────
ok_m = df_masking[(df_masking.status == 'success') &
                   df_masking.mask_path.map(lambda p: Path(p).exists())]

if ok_m.empty:
    print('⚠ Keine Masken vorhanden')
else:
    for mod in ['T1w', 'DWI']:
        sub_df = ok_m[ok_m.modality == mod]
        if sub_df.empty:
            print(f'⚠ Keine {mod}-Masken'); continue

        n_subs   = len(sub_df)
        n_slices = 5
        fig, axes = plt.subplots(n_subs, n_slices,
                                  figsize=(n_slices * 3.2, n_subs * 3.2))
        if n_subs == 1: axes = axes[np.newaxis, :]
        fig.suptitle(f'{mod} — SynthStrip Brain Masks (alle Subjects)',
                     fontsize=12, fontweight='bold', y=1.01)

        for ri, (_, r) in enumerate(sub_df.iterrows()):
            img  = nib.load(r.img_path).get_fdata(dtype=np.float32)
            mask = nib.load(r.mask_path).get_fdata(dtype=np.float32)
            if img.ndim == 4: img = img[..., 0]
            zs = np.linspace(int(img.shape[2] * 0.15),
                             int(img.shape[2] * 0.85), n_slices, dtype=int)
            brain_vox = img[mask > 0]
            vmin = brain_vox.min() if len(brain_vox) else 0
            vmax = np.percentile(brain_vox, 99) if len(brain_vox) else 1

            for ci, z in enumerate(zs):
                ax = axes[ri, ci]
                sl = np.rot90(img[:, :, z])
                ml = np.rot90(mask[:, :, z])
                ax.imshow(sl, cmap='gray', vmin=vmin, vmax=vmax)
                if ml.any(): ax.contour(ml, levels=[0.5], colors='red', linewidths=1.2)
                ax.axis('off')
                if ci == 0:
                    ax.set_ylabel(f"{r.subject}\n{r.session}", fontsize=8,
                                  rotation=0, labelpad=50, va='center')
                if ri == 0: ax.set_title(f'z={z}', fontsize=8)

        plt.tight_layout()
        fname = OUTPUT_ROOT/'qc'/f'mask_qc_{mod.lower()}_all.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'✓ {mod} Mask QC gespeichert → {fname.name}')

    # ── Histogram: Brain Volume pro Subject ───────────────────────────────────
    t1_m = ok_m[ok_m.modality == 'T1w']
    if not t1_m.empty:
        vols = []
        for _, r in t1_m.iterrows():
            mask = nib.load(r.mask_path)
            vox_vol = np.prod(mask.header.get_zooms()[:3])
            n_vox   = (mask.get_fdata() > 0).sum()
            vols.append(dict(label=f"{r.subject}\n{r.session}",
                             vol_ml=n_vox * vox_vol / 1000))
        df_vol = pd.DataFrame(vols)
        fig, ax = plt.subplots(figsize=(max(8, len(df_vol)*0.9), 4))
        bars = ax.bar(range(len(df_vol)), df_vol.vol_ml,
                      color='steelblue', alpha=0.85, edgecolor='white')
        ax.set_xticks(range(len(df_vol)))
        ax.set_xticklabels(df_vol.label, fontsize=7)
        ax.set_ylabel('Brain Volume (mL)')
        ax.set_title('T1w Brain Volume per Subject (aus SynthStrip-Maske)',
                     fontweight='bold')
        ax.axhline(df_vol.vol_ml.mean(), color='red', linestyle='--',
                   linewidth=1.2, label=f'Mean {df_vol.vol_ml.mean():.0f} mL')
        ax.legend(fontsize=8)
        for bar, v in zip(bars, df_vol.vol_ml):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                    f'{v:.0f}', ha='center', va='bottom', fontsize=7)
        plt.tight_layout()
        plt.savefig(OUTPUT_ROOT/'qc'/'brain_volume.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('✓ Brain Volume Plot gespeichert')


## 4. Data Augmentation

Random axis flips + independent Gaussian noise pairs per DWI volume.


In [ ]:
import gc

def augment_flip(vol):
    for axis in range(3):
        if np.random.rand() > 0.5:
            vol = np.flip(vol, axis=axis)
    return np.ascontiguousarray(vol)

def augment_noise(vol, noise_std_range=(0.02, 0.08)):
    std = np.random.uniform(*noise_std_range)
    ni  = (vol + np.random.normal(0, std, vol.shape)).astype(np.float32)
    nt  = (vol + np.random.normal(0, std, vol.shape)).astype(np.float32)
    return ni, nt

def znorm(v, m):
    b = v[m>0]; return (v - b.mean()) / (b.std() + 1e-8)

N_AUGMENTS = 4
aug_log    = []
np.random.seed(RANDOM_SEED)

for sub in tqdm(SUBJECTS, desc='Augmentation'):
    for ses in SESSIONS:
        rs_dwi = OUTPUT_ROOT/'resampled'/sub/ses/'dwi'
        msk_d  = OUTPUT_ROOT/'masks'/sub/ses
        aug_d  = OUTPUT_ROOT/'augmented'/sub/ses
        if not rs_dwi.exists(): continue
        aug_d.mkdir(parents=True, exist_ok=True)
        for dwi in rs_dwi.glob('*_dwi.nii.gz'):
            stem  = dwi.name.replace('.nii.gz','')
            msk   = msk_d/f'{stem}_mask.nii.gz'
            out_f = aug_d/f'{stem}_augmented.npz'
            if not msk.exists():
                print(f'  ⚠ {sub}/{ses}: no mask'); continue
            if out_f.exists():
                print(f'  [skip] {sub}/{ses}')
                d = np.load(str(out_f), mmap_mode='r')
                aug_log.append(dict(subject=sub, session=ses,
                                    n_pairs=d['inputs'].shape[0], file=str(out_f)))
                continue
            mask_data, _ = load_nifti(str(msk))
            mask_b = (mask_data > 0.5).astype(np.uint8)
            del mask_data; gc.collect()
            dwi_img = nib.load(str(dwi))
            n_vols  = dwi_img.shape[-1]
            n_per_v = N_AUGMENTS + 1
            total   = n_vols * n_per_v
            vshape  = dwi_img.shape[:3]
            all_ni  = np.zeros((total, *vshape), dtype=np.float32)
            all_nt  = np.zeros((total, *vshape), dtype=np.float32)
            for v in range(n_vols):
                vol = dwi_img.dataobj[..., v].astype(np.float32)
                vol = znorm(vol, mask_b)
                base = v * n_per_v
                ni, nt = augment_noise(vol)
                all_ni[base] = ni; all_nt[base] = nt
                for a in range(N_AUGMENTS):
                    f = augment_flip(vol.copy())
                    ni, nt = augment_noise(f)
                    all_ni[base+1+a] = ni; all_nt[base+1+a] = nt
                del vol; gc.collect()
            np.savez_compressed(str(out_f), inputs=all_ni, targets=all_nt)
            del all_ni, all_nt, mask_b; gc.collect()
            aug_log.append(dict(subject=sub, session=ses, n_pairs=total, file=str(out_f)))
            print(f'  ✓ {sub}/{ses}: {total} pairs ({n_vols} vols × {n_per_v})')

df_aug = pd.DataFrame(aug_log) if aug_log else \
         pd.DataFrame(columns=['subject','session','n_pairs','file'])
print(f'Total augmented pairs: {df_aug.n_pairs.sum():,}')


In [ ]:
# ── QC: Augmentation ─────────────────────────────────────────────────────────
aug_files = sorted((OUTPUT_ROOT/'augmented').rglob('*.npz'))[:3]
if not aug_files: print('⚠ No augmented files yet')
else:
    for af in aug_files:
        d = np.load(str(af), mmap_mode='r')
        inputs = d['inputs']; targets = d['targets']
        brain_idx = [i for i in range(min(50,len(inputs))) if inputs[i].mean()>0.01]
        idx = brain_idx[len(brain_idx)//2] if brain_idx else 0
        nz  = np.where(inputs[idx].any(axis=(0,1)))[0]
        z   = int(nz[len(nz)//2]) if len(nz) else inputs[idx].shape[2]//2
        label = af.parent.parent.name+'/'+af.parent.name
        fig, axes = plt.subplots(2, 4, figsize=(16, 8))
        fig.suptitle(f'Augmentation QC — {label}  (z={z})', fontweight='bold')
        for col in range(4):
            axes[0,col].imshow(np.rot90(inputs[col][:,:,z]), cmap='gray'); axes[0,col].axis('off')
            axes[1,col].imshow(np.rot90(targets[col][:,:,z]), cmap='gray'); axes[1,col].axis('off')
            axes[0,col].set_title(f'Input {col}', fontsize=8)
            axes[1,col].set_title(f'Target {col}', fontsize=8)
        axes[0,0].set_ylabel('Noisy Input', fontsize=8, rotation=0, labelpad=55, va='center')
        axes[1,0].set_ylabel('Noisy Target', fontsize=8, rotation=0, labelpad=55, va='center')
        plt.tight_layout()
        plt.savefig(OUTPUT_ROOT/'qc'/f'aug_qc_{label.replace("/","_")}.png', dpi=150, bbox_inches='tight')
        plt.show(); print(f'✓ {label}: {len(inputs)} pairs')


## 6. Train / Val / Test Split

In [ ]:
np.random.seed(RANDOM_SEED)
subs = df_aug.subject.unique() if 'df_aug' in dir() and len(df_aug) else np.array([])
perm = np.random.permutation(subs); n = len(perm)
n_te = max(1,round(n*.15)); n_va = max(1,round(n*.15))
train_s=list(perm[:n-n_te-n_va]); val_s=list(perm[n-n_te-n_va:n-n_te]); test_s=list(perm[n-n_te:])
if 'df_aug' in dir() and len(df_aug):
    df_aug['split']=df_aug.subject.map(lambda s:'train' if s in train_s else('val' if s in val_s else 'test'))
    print('Split:'); print(df_aug.groupby('split').agg(subjects=('subject','nunique'),pairs=('n_pairs','sum')))
    colors={'train':'#2196F3','val':'#FF9800','test':'#4CAF50'}
    split_c=df_aug.groupby('split').n_pairs.sum()
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    axes[0].pie(split_c,labels=split_c.index,autopct='%1.0f%%',
                colors=[colors.get(k,'grey') for k in split_c.index],startangle=90)
    axes[0].set_title('Augmented pairs by split')
    axes[1].bar(range(len(df_aug)),df_aug.n_pairs,
                color=[colors.get(r['split'],'grey') for _,r in df_aug.iterrows()])
    axes[1].set_xticks(range(len(df_aug)))
    axes[1].set_xticklabels([f"{r.subject}\n{r.session}" for _,r in df_aug.iterrows()],fontsize=7)
    axes[1].set_title('Pairs per subject')
    from matplotlib.patches import Patch
    axes[1].legend(handles=[Patch(color=v,label=k) for k,v in colors.items()])
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT/'qc'/'split_qc.png',dpi=150,bbox_inches='tight'); plt.show()
    with open(OUTPUT_ROOT/'split_manifest.json','w') as f:
        json.dump(dict(train=list(train_s),val=list(val_s),test=list(test_s),
            files=df_aug[['subject','session','split','file']].to_dict('records')),f,indent=2)
    print('✓ Split manifest saved')


## 7. Pipeline Summary

In [ ]:
print('='*52)
print('    MRI DATA PREPARATION — FINAL SUMMARY')
print('='*52)
for label,val in [
    ('Subjects',        len(SUBJECTS)),
    ('Target shape',    TARGET_SHAPE),
    ('Resampled',       (df_rs.status=='success').sum()     if 'df_rs'      in dir() else '—'),
    ('Masks',           (df_masking.status=='success').sum() if 'df_masking' in dir() else '—'),
    ('Augmented pairs', int(df_aug.n_pairs.sum())            if 'df_aug'     in dir() and len(df_aug) else '—'),
    ('Train subjects',  train_s if 'train_s' in dir() else '—'),
    ('Val subjects',    val_s   if 'val_s'   in dir() else '—'),
    ('Test subjects',   test_s  if 'test_s'  in dir() else '—'),
]:
    print(f'  {label:<22}: {val}')
print('='*52)
